In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt # plotting 
import seaborn # another plotting libary, nice visualizations

In [2]:
path_to_data = "/Users/nehiryuksekkaya/Desktop/are-sd-2024-emissions-co2-du-mix-electrique/"

data = pd.read_csv(path_to_data + "train.csv")

In [3]:
data = pd.read_csv(path_to_data + 'train.csv', index_col='DateTime')
prev_data = pd.read_csv(path_to_data + 'prev.csv', index_col='DateTime')

In [4]:
train=data[:int((len(data)*1)*.7)+1]
test=data[int((len(data)*1)*.3):]

In [5]:
## First an utility function, how can we compute distances from strings? 
## we need a label encoder, which already exists in sklearn for example, 
## here we implement it from scratch

def label_encoder(feature_vec):
    """
    Encoding categorical features / labels (e.g. text) to integers
    """
    unique_labels = set(feature_vec)
    index = [i for i in range(len(unique_labels))]
    label_to_index = {label: index for (label,index) in zip(unique_labels,index)}
    encoded_labels = [label_to_index[label] for label in feature_vec]
    return np.array(encoded_labels)

In [6]:
## kNN works by first establishing a distance function, here are two 
## but I'm sure you can come up with more

def modulo_distance(xTrain, xTest, modulo = 7):
    """
    Generalized modulo distance, when could this one be useful for our 
    dataset? anything that repeats regularly amond the features? 
    """
    diff = np.abs(xTrain[:, np.newaxis] - xTest)
    squared_distance = (modulo % (diff +1e-7))**2
    distances = squared_distance
    distances[distances < 0] = 0
    distances = np.sqrt(distances)
    return distances


def euclidean_distance(xTrain, xTest):
    """
    When the euclidean distance can be useful indeed? 
    """
    distances = np.sqrt(np.sum((xTrain[:, np.newaxis] - xTest) ** 2, axis=-1))
    distances[distances < 0] = 0
    
    return distance



def my_distance1(xTrain, xTest):
    """
    Can you suggest another one? 
    Double check your TME notebooks from the Data Science course
    """
    distances = np.sum(np.abs(xTrain[:, np.newaxis] - xTest) ** p, axis=-1) ** (1/p)
    distances[distances < 0] = 0
    return distances 

In [7]:
def knn(xTrain, xTest, k, modulo = None):
    """
    Finds the k nearest neighbors of xTest in xTrain.
    
    -----
    Input:
    xTrain = n x d matrix. n=rows and d=features
    xTest = m x d matrix. m=rows and d=features (same amount of features as xTrain)
    k = number of nearest neighbors to be found
    
    Output:
    dists = distances between xTrain/xTest points. Size of n x m
    indices = kxm matrix with indices of yTrain labels
    """
    
    #None
    if modulo is None:
        # Calculate distances without modulo operation
        distances = np.sqrt(np.sum((xTrain[:, np.newaxis] - xTest) ** 2, axis=-1))
    else:
        # Calculate distances with modulo operation
        diff = np.abs(xTrain[:, np.newaxis] - xTest)
        squared_distance = (modulo % (diff + 1e-7)) ** 2
        distances = np.sqrt(squared_distance)
    
    # Find indices of k nearest neighbors
    indices = np.argsort(distances, axis=0)[:k, :]
    
    return distances, indices

def knn_classification(xTrain, yTrain, xTest, k=3, modulo = None):
    """
    Builds a classifier after getting distances
    
    
    -----
    Input:
    xTrain = n x d matrix. n=rows and d=features
    yTrain = n x 1 array. n=rows with label value
    xTest = m x d matrix. m=rows and d=features
    k = number of nearest neighbors to be found
    
    Output:
    predictions = predicted labels
    
    """
    # Get indices and distances of k nearest neighbors
    indices, distances = knn(xTrain, xTest, k, modulo)
    
    # Get labels of k nearest neighbors
    nearest_labels = yTrain[indices]
    
    # Perform majority voting to determine predicted labels
    predictions, _ = mode(nearest_labels, axis=0)
    
    return predictions

In [17]:
def split_train_test(X,y,ratio_train=0.8):  #It can also be done randomly, here we decided to choose ourselves
    X_train= X[:int(ratio_train*len(X))]
    X_test= X[int(ratio_train*len(X)):]
    y_train= y[:int(ratio_train*len(y))]
    y_test = y[int(ratio_train*len(y)):]
    return X_train,X_test,y_train,y_test

def train_val_split(X, y, val_ratio = 0.2, shuffle = False):
    """
    Perform a simple train-validation split on the given dataset.
    
    Parameters:
        X (numpy.ndarray): The input features of shape (n_samples, n_features).
        y (numpy.ndarray): The corresponding class labels of shape (n_samples,).
        val_ratio (float): The ratio of validation data to total data (0 < val_ratio < 1).
        shuffle (bool): whether to shuffle the data before splitting.
    
    Returns:
        X_train (numpy.ndarray): The input features of the training set.
        X_val (numpy.ndarray): The input features of the validation set.
        y_train (numpy.ndarray): The class labels of the training set.
        y_val (numpy.ndarray): The class labels of the validation set.
    """
    # Get the number of samples
    n_samples = X.shape[0]
    
    # Shuffle the data if required
    if shuffle:
        indices = np.random.permutation(n_samples)
        X = X[indices]
        y = y[indices]
    
    # Calculate the number of samples for the validation set
    n_val_samples = int(n_samples * val_ratio)
    
    # Split the data into training and validation sets
    X_train, X_val = X[:-n_val_samples], X[-n_val_samples:]
    y_train, y_val = y[:-n_val_samples], y[-n_val_samples:]
    
    return X_train, X_val, y_train, y_val


def kfold_cross_validation(X, k, shuffle = False):
    """
    Perform standard k-fold cross-validation on the given dataset.
    
    Parameters:
        X (numpy.ndarray): The input features of shape (n_samples, n_features).
        k (int): The number of folds for cross-validation.
        shuffle (bool): whether to shuffle the data before splitting.
    
    Yields:
        train_indices (numpy.ndarray): The indices of the training set for the current fold.
        val_indices (numpy.ndarray): The indices of the validation set for the current fold.
    """
   
    # Get the number of samples
    n_samples = X.shape[0]

    # Generate indices for cross-validation
    indices = np.arange(n_samples)
    if shuffle:
        np.random.shuffle(indices)
    
    # Calculate the size of each fold
    fold_size = n_samples // k
    
    # Split data into k folds
    for i in range(k):
        val_start = i * fold_size
        val_end = (i + 1) * fold_size
        
        # Get validation indices
        val_indices = indices[val_start:val_end]
        
        # Get training indices
        train_indices = np.concatenate((indices[:val_start], indices[val_end:]))
        yield train_indices, val_indices
    

In [9]:
X_train = train.iloc[:, :-1]  # Features for training
y_train = train.iloc[:, -1]   # Labels for training

xTest = test.iloc[:, :-1]    # Features for testing
yTest = test.iloc[:, -1]     # Labels for testing

In [18]:
X_train,X_test,y_train,y_test=split_train_test(data,data['MixProdElec'],ratio_train=0.8)

In [10]:
# Example Usage of train validation split 

xtrain,  xval, ytrain, yval = train_val_split(X_train, y_train, val_ratio = 0.2, shuffle = False)

In [19]:
# do you know what tqdm is? a quick way to tell how much each iteration of the for loop 
# takes through a progress bar (check it on google)
from tqdm.notebook import tqdm
# Example Usage of cross validation split 
# it has to be in a for loop (think about the nature of cross-validation)

k = 3
counter = 0
for train_indices, val_indices in tqdm(kfold_cross_validation(X_train, k, shuffle= False), total = k):
    
    print("Fold # " + str(counter))
    # print(train_indices)
    # print("\n")
    xtrain, xval = X_train[train_indices], X_train[val_indices]
    ytrain, yval = y_train[train_indices], y_train[val_indices]
    counter = counter + 1
    # Perform your training and evaluation using the train and validation sets
    # ...

  0%|          | 0/3 [00:00<?, ?it/s]

Fold # 0


KeyError: "None of [Index([25596, 25597, 25598, 25599, 25600, 25601, 25602, 25603, 25604, 25605,\n       ...\n       76778, 76779, 76780, 76781, 76782, 76783, 76784, 76785, 76786, 76787],\n      dtype='int64', length=51192)] are in the [columns]"

In [21]:
prev = knn_classification(X_train,y_train,X_test, k=5, modulo = None)

InvalidIndexError: (slice(None, None, None), None)

In [14]:
submission = pd.read_csv(path_to_data + 'benchmark.csv',index_col='DateTime').copy()
submission['MixProdElec'] = prev
submission.to_csv('submission_bayes.csv', index=True, index_label='DateTime')

InvalidIndexError: (slice(None, None, None), None)